## Cleaning the Fed Speech data

In [22]:
import pandas as pd
import os 

In [23]:
# print(os.getcwd())
speeches = pd.read_csv("../data/us_speeches.csv")

# speeches
# type(speeches)

### first, check average word count as is
Then, clean each of the different sources of their headers, intros, etc. to get just the raw speech content

In [24]:
speeches["word_count"] = speeches["text"].apply(lambda x: len(x.split()))
speeches.word_count.mean()

np.float64(2950.394430149841)

In [25]:
speeches.loc[speeches["Source"] == "CB websites"].word_count.mean()

np.float64(2813.1871737509323)

In [26]:
speeches.Source.value_counts()

Source
CB websites    2682
BIS            2223
Archives       1702
Name: count, dtype: int64

### Now we clean the "CB websites" first

In [27]:
# select only cb source speeches
cb = speeches.loc[speeches["Source"] == "CB websites"]

# create a parsed date variable which includes the date in the format "Month Day, Year" for each row
# but in case the date is 2008-01-08, it should be converted to "January 8, 2008", not "January 08, 2008"
cb["parsed_date"] = cb.apply(lambda row: pd.to_datetime(row["Date"]).strftime("%B %-d, %Y"), axis=1)

# create a new column text-cleaned which includes only the text after the parsed date is mentioned in the text column for the first time
cb["text-cleaned"] = cb.apply(lambda row: row["text"].split(row["parsed_date"])[-1] if row["parsed_date"] in row["text"] else row["text"], axis=1)
# cb["text-cleaned"] = cb.apply(lambda row: row["text"].split(row["parsed_date"])[-1], axis=1)

# in case text stats with "Last Updated", remove everything before and incl. "Introduction"
cb["text-cleaned"] = cb.apply(lambda row: row["text-cleaned"].split("Introduction")[-1] if "Last Updated" in row["text-cleaned"] else row["text-cleaned"], axis=1)

Ok, some of these datasets still are uncleaned, i.e. those that start with:

i.e. if it starts with "Last Updated" remove until "Introduction"

## remove end of speech 

patterns to match:
-> "Thank you. NOTES:" 


In [28]:
# remove the part of the text, after "Thank you. NOTES:"
cb["text-cleaned"] = cb.apply(lambda row: row["text-cleaned"].split("Thank you. NOTES:")[0] if "Thank you. NOTES:" in row["text-cleaned"] else row["text-cleaned"], axis=1)

In [29]:
# recalculate word count for the cleaned text
cb["word_count_cl"] = cb["text-cleaned"].apply(lambda x: len(x.split()))
cb["word_count_dirt"] = cb["text"].apply(lambda x: len(x.split()))

cb["word_count_cl"].mean()
cb["word_count_dirt"].mean()

np.float64(2813.1871737509323)

In [30]:
# to the cb dataframe, add a column called "words-removed" which incl. the difference in word count between the original text and the cleaned text
cb["words-removed"] = cb["word_count_dirt"] - cb["word_count_cl"]
cb["words-removed"].head()

2223    26
2224    17
2225    34
2226    88
2227    13
Name: words-removed, dtype: int64

In [31]:
# if word_count_cl is less than 100, then replace text-cleaned with the original text
cb["text-cleaned"] = cb.apply(lambda row: row["text"] if row["word_count_cl"] < 100 else row["text-cleaned"], axis=1)
cb["word_count_cl"] = cb["text-cleaned"].apply(lambda x: len(x.split()))
cb["words-removed"] = cb["word_count_dirt"] - cb["word_count_cl"]


### Clean BIS data 

In [33]:
speeches.loc[speeches["Source"] == "BIS"].word_count.mean()

np.float64(3219.3288349077825)

In [34]:
# select only cb source speeches
bis = speeches.loc[speeches["Source"] == "BIS"]

# bis has "* * *" in the text, remove everything before and incl. "* * *"
bis["text-cleaned"] = bis.apply(lambda row: row["text"].split("* * * ")[-1] if "* * * " in row["text"] else row["text"], axis=1)
bis["word_count_cl"] = bis["text-cleaned"].apply(lambda x: len(x.split()))
bis["word_count_dirt"] = bis["text"].apply(lambda x: len(x.split()))
bis["words-removed"] = bis["word_count_dirt"] - bis["word_count_cl"]
bis["words-removed"].head()
# bis.to_csv("../data/bis_speeches.csv", index=False)

0    46
1    41
2    38
3    56
4    43
Name: words-removed, dtype: int64

### Clean Archive Dataset


In [35]:
arch = speeches.loc[speeches["Source"] == "Archives"]

arch["parsed_date"] = arch.apply(lambda row: pd.to_datetime(row["Date"]).strftime("%B %-d, %Y"), axis=1)

# create a new column text-cleaned which includes only the text after the parsed date is mentioned in the text column for the first time
arch["text-cleaned"] = arch.apply(lambda row: row["text"].split(row["parsed_date"])[-1], axis=1)
# cb["text-cleaned"] = cb.apply(lambda row: row["text"].split(row["parsed_date"])[-1], axis=1)
arch["word_count_cl"] = arch["text-cleaned"].apply(lambda x: len(x.split()))
arch["word_count_dirt"] = arch["text"].apply(lambda x: len(x.split()))
arch["words-removed"] = arch["word_count_dirt"] - arch["word_count_cl"]
arch["words-removed"].head()
# arch.to_csv("../data/arch_speeches.csv", index=False)
arch



# if word_count_cl is less than 4, then replace text-cleaned with the original text
arch["text-cleaned"] = arch.apply(lambda row: row["text"] if row["word_count_cl"] < 4 else row["text-cleaned"], axis=1)
arch["word_count_cl"] = arch["text-cleaned"].apply(lambda x: len(x.split()))
arch["words-removed"] = arch["word_count_dirt"] - arch["word_count_cl"]
arch["words-removed"].head()
# arch.to_csv("../data/arch_speeches.csv", index=False)

3396    44
3397    48
3398    35
3399    38
3400    53
Name: words-removed, dtype: int64

In [36]:
# now reconcatentate all 3 dataframes into one
final_speeches = pd.concat([cb, bis, arch], ignore_index=True)
final_speeches.to_csv("../data/cleaned_speeches.csv", index=False)